# AutoDataLab++ — CoS evaluation on Kaggle GPU

Runs the **same evaluation idea** as `cos_grpo_colab.ipynb` / the CoS eval Space: load a **base model** (+ optional **LoRA** from the Hub), take the model’s **first JSON action**, then **finish the episode** with a deterministic continuation so the **terminal grader** score matches your local pipeline.

**Setup**
1. **Settings → Accelerator → GPU** (T4 is enough for 1.5B / small LoRA).
2. **Settings → Internet → On** (for `pip`, `git clone`, and Hub weights).
3. Either add this repo as a **Kaggle Dataset** and set `ENV_LOCAL_PATH` below, **or** set `ENV_REPO_URL` to `git clone` into `/kaggle/working`.
4. **HF token**: paste in the config cell, or add a Kaggle secret named `HF_TOKEN` (read access is enough for public weights; **write** if you push adapters).

No Hugging Face **Space** is required — the env runs **in this notebook**.

**Noise you can ignore:** lines like `Unable to register cuFFT/cuDNN factory` or `computation placer already registered` usually mean TensorFlow/JAX and PyTorch both touched CUDA in the same process — they do not stop training.

**If you see** `No module named 'triton.backends'` **or** `cannot import name 'ir' from 'triton._C.libtriton'`: it is a **torch ↔ triton ABI mismatch** triggered by `torchvision`. Run the install cell below (it removes `torchvision`, which we do not need for LLM inference), then **Session → Restart session** and run all cells from the top.

In [ ]:
# We do NOT upgrade torch/torchvision/triton on Kaggle (mismatched ABIs cause:
#   ImportError: cannot import name 'ir' from 'triton._C.libtriton'
#   AttributeError: module 'triton' has no attribute 'backends'
# transformers triggers torchvision → torch._dynamo → triton on import. We only need
# inference (no compile, no vision), so the safest fix is to remove torchvision.
import subprocess
import sys

_PY = sys.executable
_PKGS = [
    "transformers>=4.45,<4.49",
    "peft>=0.13,<0.16",                # adapters trained on newer peft set fields like eva_config
    "accelerate>=0.33,<1.1",
    "bitsandbytes>=0.45.0",            # Kaggle CUDA 12.8 needs newer bnb wheel
    "huggingface_hub>=0.24,<1.0",
    "pydantic>=2",
    "tqdm",
    "pandas",
]
subprocess.check_call(
    [_PY, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + _PKGS,
)
subprocess.run([_PY, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
print("pip ok — now: Session → Restart session, then Run All from the top.")

## 1) Config — HF token, model weights, env location

In [ ]:
import os
from pathlib import Path

# --- Hugging Face token (optional if all repos/models are public) ---
HF_TOKEN = ""  # paste here, OR leave empty and set Kaggle secret "HF_TOKEN"

try:
    from kaggle_secrets import UserSecretsClient
    if not HF_TOKEN.strip():
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

os.environ["HF_TOKEN"] = HF_TOKEN or ""
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN or ""

# --- Model weights ---
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # base checkpoint on the Hub
ADAPTER_ID = ""  # optional: e.g. "your-user/your-lora-repo"
ADAPTER_SUBFOLDER = ""  # optional subfolder inside the adapter repo (e.g. "final")
USE_4BIT = True  # set False if load fails or you use A100 with headroom

# --- Env code (AutoDataLab++ root with ceo_brief_env/) ---
# Option A: dataset mounted at /kaggle/input/your-dataset-name/autodatalab-plus
ENV_LOCAL_PATH = "/kaggle/input/autodatalab-plus"  # change if you zip-uploaded the repo
# Option B: git clone if A is missing
ENV_REPO_URL = "https://github.com/Uchihakamal1816/AutoDataLab-.git"  # TODO: your fork with ceo_brief_env
ENV_REPO_REF = "main"
ENV_CLONE_DIR = Path("/kaggle/working/autodatalab-plus")

# --- Eval ---
TASKS = ["easy_brief", "medium_brief", "hard_brief", "expert_brief"]
EPISODES_PER_TASK = 3
USE_RAG = False

print("MODEL_ID:", MODEL_ID)
print("ADAPTER_ID:", ADAPTER_ID or "(none)")
print("HF_TOKEN set:", bool((HF_TOKEN or "").strip()))

## 2) Put `ceo_brief_env` on `sys.path` (clone if needed) + `pip install -e`

In [ ]:
import subprocess
import sys

def resolve_env_root() -> Path:
    p = Path(ENV_LOCAL_PATH)
    if p.is_dir() and (p / "ceo_brief_env").is_dir():
        return p.resolve()
    ENV_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    if ENV_CLONE_DIR.is_dir():
        subprocess.run(["rm", "-rf", str(ENV_CLONE_DIR)], check=False)
    cmd = ["git", "clone", "--depth", "1", "-b", ENV_REPO_REF, ENV_REPO_URL, str(ENV_CLONE_DIR)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        cmd2 = ["git", "clone", "--depth", "1", ENV_REPO_URL, str(ENV_CLONE_DIR)]
        r2 = subprocess.run(cmd2, capture_output=True, text=True)
        if r2.returncode != 0:
            raise RuntimeError(f"git clone failed:\n{r.stderr}\n{r2.stderr}")
    root = ENV_CLONE_DIR.resolve()
    if not (root / "ceo_brief_env").is_dir():
        raise RuntimeError(f"No ceo_brief_env under {root}")
    return root

ENV_ROOT = resolve_env_root()
if str(ENV_ROOT) not in sys.path:
    sys.path.insert(0, str(ENV_ROOT))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(ENV_ROOT)],
    check=True,
)
print("ENV_ROOT:", ENV_ROOT)

## 3) Load model + run evaluation (first action from LLM, then deterministic finish)

In [ ]:
import gc
import json
import os
import re
from typing import Any

# Before importing torch: avoid compile paths that hard-require triton on some images
os.environ.setdefault("TORCH_COMPILE_DISABLE", "1")
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from ceo_brief_env.environment import CEOBriefEnvironment, required_experts_for_task
from ceo_brief_env.models import CoSAction, CoSObservation

VALID_ACTIONS = {"consult", "ask", "summarize", "submit", "noop"}
VALID_EXPERTS = {"analyst", "finance", "hr", "strategy"}
_JSON_RE = re.compile(r"\{[^{}]*\}", re.S)

SYSTEM_PROMPT = (
    "You are the Chief of Staff in AutoDataLab++. You orchestrate four specialists: "
    "analyst, finance, strategy, hr. Reply with STRICT JSON only.\n"
    'Schema: {"action_type": one of [consult, ask, summarize, submit, noop], '
    '"expert_id": one of [analyst, finance, hr, strategy] or null}.\n'
    "Rules: consult each required expert at most once -> summarize -> submit."
)


def render_obs(obs: CoSObservation) -> str:
    return (
        f"task={obs.task_name} step={obs.step_count}/{obs.max_steps} "
        f"rag={obs.rag_enabled} consulted={obs.consulted_experts} "
        f"brief_done={obs.current_brief is not None} available={obs.available_experts}"
    )


def parse_action(text: str) -> CoSAction:
    m = _JSON_RE.search(text or "")
    if not m:
        return CoSAction(action_type="noop")
    try:
        a = json.loads(m.group(0))
    except Exception:
        return CoSAction(action_type="noop")
    at = a.get("action_type")
    if at not in VALID_ACTIONS:
        return CoSAction(action_type="noop")
    eid = a.get("expert_id")
    if eid is not None and eid not in VALID_EXPERTS:
        eid = None
    return CoSAction(action_type=at, expert_id=eid)


def deterministic_continuation(env: CEOBriefEnvironment, obs: CoSObservation, task: str) -> float:
    while not obs.done and obs.step_count < obs.max_steps:
        missing = [e for e in required_experts_for_task(task) if e not in obs.consulted_experts]
        if missing:
            act = CoSAction(action_type="consult", expert_id=missing[0])  # type: ignore[arg-type]
        elif obs.current_brief is None:
            act = CoSAction(action_type="summarize")
        else:
            act = CoSAction(action_type="submit")
        obs = env.step(act)
    return float(obs.terminal_grader_score or 0.0)


def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or None)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = None
    if USE_4BIT:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            token=HF_TOKEN or None,
            device_map="auto",
            quantization_config=bnb,
            torch_dtype=torch.bfloat16,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            token=HF_TOKEN or None,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
    model.eval()
    if (ADAPTER_ID or "").strip():
        from peft import PeftModel

        kw: dict[str, Any] = {"token": HF_TOKEN or None}
        if (ADAPTER_SUBFOLDER or "").strip():
            kw["subfolder"] = ADAPTER_SUBFOLDER.strip()
        model = PeftModel.from_pretrained(model, ADAPTER_ID.strip(), **kw)
        model.eval()
    return tok, model


@torch.no_grad()
def generate_action(model, tok, obs: CoSObservation):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": render_obs(obs)},
    ]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    gen = model.generate(
        **ids,
        max_new_tokens=48,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )
    comp = tok.decode(gen[0, ids.input_ids.shape[1] :], skip_special_tokens=True)
    return parse_action(comp), comp.strip()[:300]


def run_evaluation():
    tok, model = load_model()
    rows = []
    raw = []
    for task in TASKS:
        scores = []
        for ep in tqdm(range(EPISODES_PER_TASK), desc=task):
            env = CEOBriefEnvironment()
            obs = env.reset(task=task, use_rag=USE_RAG)
            try:
                action, completion = generate_action(model, tok, obs)
                obs = env.step(action)
                term = deterministic_continuation(env, obs, task)
            except Exception as e:
                completion = f"<error: {e}>"
                term = 0.0
            scores.append(term)
            raw.append(
                {
                    "task": task,
                    "episode": ep,
                    "first_action": action.model_dump(exclude_none=True),
                    "completion_preview": completion,
                    "terminal": round(float(term), 4),
                }
            )
        mean = round(sum(scores) / len(scores), 4)
        rows.append({"task": task, "episodes": EPISODES_PER_TASK, "mean_terminal": mean, "scores": scores})
    overall = round(sum(r["mean_terminal"] for r in rows) / len(rows), 4)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows, raw, overall


eval_rows, eval_raw, mean_overall = run_evaluation()
import pandas as pd
from IPython.display import display

display(pd.DataFrame(eval_rows))
print("mean_overall (avg of per-task means):", mean_overall)
print("--- sample raw traces (first 3) ---")
for x in eval_raw[:3]:
    print(x)

## 4) (Optional) Save results JSON to `/kaggle/working` for download

In [ ]:
out = {
    "model_id": MODEL_ID,
    "adapter_id": ADAPTER_ID or None,
    "mean_overall": mean_overall,
    "per_task": eval_rows,
    "raw": eval_raw,
}
p = Path("/kaggle/working/cos_eval_results.json")
p.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("Wrote", p)